In [1]:
import random
import pandas as pd
import json
random.seed(707)
n = 400
lokasi_list = ["Cabang Magelang", "Cabang Semarang", "Cabang Yogyakarta"]
kategori_dirty = ["laptop","Laptop","LAPTOP","hp","Hp","HP", "tablet","Tablet"]

teknisi_ids = [f"TK{i:03d}" for i in range(1, 20)]
lokasi_per_teknisi = {kid: random.choice(lokasi_list) for kid in teknisi_ids}

rows = []
for i in range(n):
    teknisi_id = random.choice(teknisi_ids)
    rows.append({
        "servis_id": f"SVS-{1000+i}",
        "teknisi_id": teknisi_id,
        "cabang": lokasi_per_teknisi[teknisi_id],
        "jenis_perangkat": random.choice(kategori_dirty),
        "durasi_servis_menit": random.randint(15, 120),
        "biaya_servis": random.choice(
            [75000,100000,150000,200000,350000]),
        "status_servis": random.choice(
            ["Selesai","Selesai","Selesai","Batal"]),
    })
    
rows += [rows[5], rows[24], rows[88]]
for idx in [7, 43, 148, 188]:
    rows[idx] = dict(rows[idx])
    rows[idx]["durasi_servis_menit"] = None
for idx in [11, 98]:
    rows[idx] = dict(rows[idx])
    rows[idx]["biaya_servis"] = None
for idx in [228, 278]:
    rows[idx] = dict(rows[idx])
    rows[idx]["durasi_servis_menit"] = -random.randint(1,10)
    
pd.DataFrame(rows).to_csv("servis.csv", index=False)
teknisi_master = [
    {"teknisi_id": kid, "nama_teknisi": f"Teknisi-{kid}",
    "cabang": lokasi_per_teknisi[kid],
    "rating_teknisi": round(random.uniform(3.0,5.0),1)}
    for kid in teknisi_ids
]

with open("teknisi.json", "w") as f:
    for k in teknisi_master:
        f.write(json.dumps(k) + "\n")
        
print("Data berhasil dibuat: servis.csv dan teknisi.json")

Data berhasil dibuat: servis.csv dan teknisi.json


In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ResponsiUTS").master("local[*]").getOrCreate()

print("SparkSession berhasil dibuat: ", spark.version)

26/10/09 15:56:51 WARN Utils: Your hostname, mazedev-1701 resolves to a loopback address: 127.0.1.1; using 192.168.18.87 instead (on interface wlp2s0)
26/10/09 15:56:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/09 15:56:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat:  3.5.9


In [3]:
# Membuat folder baru di hdfs
!hdfs dfs -mkdir -p "/user/mazedev/uts/raw/"

# Upload file ke hdfs
!hdfs dfs -put -f "servis.csv" "/user/mazedev/uts/raw/"

# Verifikasi file
!hdfs dfs -ls "/user/mazedev/uts/raw/"

Found 1 items
-rw-r--r--   1 mazedev supergroup      23732 2026-10-08 08:41 /user/mazedev/uts/raw/servis.csv


In [5]:
# Baca servis.csv dari HDFS
df_servis = spark.read.csv("hdfs://localhost:9000/user/mazedev/uts/raw/servis.csv", header=True, inferSchema=True)
df_servis.printSchema()

root
 |-- servis_id: string (nullable = true)
 |-- teknisi_id: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- jenis_perangkat: string (nullable = true)
 |-- durasi_servis_menit: double (nullable = true)
 |-- biaya_servis: double (nullable = true)
 |-- status_servis: string (nullable = true)



In [6]:
# Baca teknisi.json dari lokal
df_teknisi = spark.read.json("teknisi.json")
df_teknisi.printSchema()

root
 |-- cabang: string (nullable = true)
 |-- nama_teknisi: string (nullable = true)
 |-- rating_teknisi: double (nullable = true)
 |-- teknisi_id: string (nullable = true)



In [7]:
# Menghitung baris
print("Jumlah baris df_servis: ", df_servis.count())
print("Jumlah baris df_teknisi: ", df_teknisi.count())

Jumlah baris df_servis:  403
Jumlah baris df_teknisi:  19


In [8]:
# Manampilkan bukti bahwa servis.csv masih kotor
df_servis.show()

+---------+----------+-----------------+---------------+-------------------+------------+-------------+
|servis_id|teknisi_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis|
+---------+----------+-----------------+---------------+-------------------+------------+-------------+
| SVS-1000|     TK003|Cabang Yogyakarta|             Hp|              113.0|    200000.0|      Selesai|
| SVS-1001|     TK009|Cabang Yogyakarta|         Tablet|               50.0|    200000.0|        Batal|
| SVS-1002|     TK008|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|
| SVS-1003|     TK016|  Cabang Magelang|         Tablet|               27.0|    200000.0|        Batal|
| SVS-1004|     TK003|Cabang Yogyakarta|         Tablet|               76.0|    200000.0|      Selesai|
| SVS-1005|     TK001|  Cabang Magelang|         Tablet|               95.0|    150000.0|      Selesai|
| SVS-1006|     TK010|  Cabang Magelang|         laptop|        

Dari hasil data yang ditampilkan diatas, ada beberapa data yang harus diperbaiki:
1. Data jenis_perangkat masih campur ada yang lower semua ada yang campuran
2. Data durasi_servis_menit masih memiliki nilai NULL
3. Data status_servis masih memiliki nilai NULL

In [8]:
# Menyamakan data jenis_perangkat agar lower
from pyspark.sql.functions import lower

df_servis = df_servis.withColumn("jenis_perangkat", lower(df_servis["jenis_perangkat"]))

df_servis.show()

+---------+----------+-----------------+---------------+-------------------+------------+-------------+
|servis_id|teknisi_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis|
+---------+----------+-----------------+---------------+-------------------+------------+-------------+
| SVS-1000|     TK003|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|
| SVS-1001|     TK009|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|
| SVS-1002|     TK008|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|
| SVS-1003|     TK016|  Cabang Magelang|         tablet|               27.0|    200000.0|        Batal|
| SVS-1004|     TK003|Cabang Yogyakarta|         tablet|               76.0|    200000.0|      Selesai|
| SVS-1005|     TK001|  Cabang Magelang|         tablet|               95.0|    150000.0|      Selesai|
| SVS-1006|     TK010|  Cabang Magelang|         laptop|        

In [10]:
# Mengganti nilai NULL pada durasi_servis_menit dan biaya_servis menjadi 0.0
# Menggunakan ini karena ingin mempertahankan baris data pada data yagn terdapat nilai NULL agar tidak hilang
# Karena jika hilang, dapat mempengaruhi data jumlah servis yang terjadi

df_servis_clean = df_servis.na.fill(0.0, subset=["durasi_servis_menit", "biaya_servis"])
df_servis_clean.show()

+---------+----------+-----------------+---------------+-------------------+------------+-------------+
|servis_id|teknisi_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis|
+---------+----------+-----------------+---------------+-------------------+------------+-------------+
| SVS-1000|     TK003|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|
| SVS-1001|     TK009|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|
| SVS-1002|     TK008|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|
| SVS-1003|     TK016|  Cabang Magelang|         tablet|               27.0|    200000.0|        Batal|
| SVS-1004|     TK003|Cabang Yogyakarta|         tablet|               76.0|    200000.0|      Selesai|
| SVS-1005|     TK001|  Cabang Magelang|         tablet|               95.0|    150000.0|      Selesai|
| SVS-1006|     TK010|  Cabang Magelang|         laptop|        

In [11]:
# tampilkan jumlah baris sesudah sebagai bukti
print("Jumlah baris df_servis sesudah cleaning: ", df_servis_clean.count())

Jumlah baris df_servis sesudah cleaning:  403


Dari jumlah baris data sesudah cleaning tetap sama yaitu 403, masih sama dengan sebelum kita melakukan cleaning di cell 17. 

In [12]:
# Transformasi data dan Penggabungan
df_gabungan = df_servis_clean.join(df_teknisi.drop("cabang"), on="teknisi_id", how="left")

#Kita drop kolom cabang di df_teknisi saat join agar tidak ada duplikasi di df_gabungan

df_gabungan.show()

+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+
|teknisi_id|servis_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis| nama_teknisi|rating_teknisi|
+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+
|     TK003| SVS-1000|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|Teknisi-TK003|           3.8|
|     TK009| SVS-1001|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|Teknisi-TK009|           3.2|
|     TK008| SVS-1002|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|Teknisi-TK008|           4.0|
|     TK016| SVS-1003|  Cabang Magelang|         tablet|               27.0|    200000.0|        Batal|Teknisi-TK016|           3.5|
|     TK003| SVS-1004|Cabang Yogyakarta|         tablet|             

In [13]:
# Menambahkan kolom baru kategori_layanan
from pyspark.sql.functions import col, when

df_gabungan = df_gabungan.withColumn(
    "kategori_layanan",
    when(col("durasi_servis_menit") <= 60, "Sesuai Target").otherwise("Melebihi Target")
)

df_gabungan.show()

+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+
|teknisi_id|servis_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis| nama_teknisi|rating_teknisi|kategori_layanan|
+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+
|     TK003| SVS-1000|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|Teknisi-TK003|           3.8| Melebihi Target|
|     TK009| SVS-1001|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|Teknisi-TK009|           3.2|   Sesuai Target|
|     TK008| SVS-1002|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|Teknisi-TK008|           4.0| Melebihi Target|
|     TK016| SVS-1003|  Cabang Magelang|         tablet|               27.0|    200000.0|        Bat

In [14]:
# Menambahkan kolom biaya_per_menit_servis
df_gabungan = df_gabungan.withColumn(
    "biaya_per_menit_servis",
    col("biaya_servis") / col("durasi_servis_menit")
)

df_gabungan.show()

+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+----------------------+
|teknisi_id|servis_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis| nama_teknisi|rating_teknisi|kategori_layanan|biaya_per_menit_servis|
+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+----------------------+
|     TK003| SVS-1000|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|Teknisi-TK003|           3.8| Melebihi Target|    1769.9115044247787|
|     TK009| SVS-1001|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|Teknisi-TK009|           3.2|   Sesuai Target|                4000.0|
|     TK008| SVS-1002|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|Teknisi-TK008|           4.0| Me

In [15]:
# Check NULL
df_check = df_gabungan.filter(col("biaya_per_menit_servis").isNull())
print(df_check.count())

4


In [16]:
# Karena ada data NULL akibat dari angka 0 yang dibagi dengan angka lain, kita fill dulu
df_gabungan = df_gabungan.na.fill(0.0, subset=["biaya_per_menit_servis"])

df_gabungan.show()

# Verifikasi lagi
df_check = df_gabungan.filter(col("biaya_per_menit_servis").isNull())
print(df_check.count())

+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+----------------------+
|teknisi_id|servis_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis| nama_teknisi|rating_teknisi|kategori_layanan|biaya_per_menit_servis|
+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+----------------------+
|     TK003| SVS-1000|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|Teknisi-TK003|           3.8| Melebihi Target|    1769.9115044247787|
|     TK009| SVS-1001|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|Teknisi-TK009|           3.2|   Sesuai Target|                4000.0|
|     TK008| SVS-1002|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|Teknisi-TK008|           4.0| Me

In [17]:
# Menampilkan 10 baris hasil akhir
df_gabungan.show(10)

+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+----------------------+
|teknisi_id|servis_id|           cabang|jenis_perangkat|durasi_servis_menit|biaya_servis|status_servis| nama_teknisi|rating_teknisi|kategori_layanan|biaya_per_menit_servis|
+----------+---------+-----------------+---------------+-------------------+------------+-------------+-------------+--------------+----------------+----------------------+
|     TK003| SVS-1000|Cabang Yogyakarta|             hp|              113.0|    200000.0|      Selesai|Teknisi-TK003|           3.8| Melebihi Target|    1769.9115044247787|
|     TK009| SVS-1001|Cabang Yogyakarta|         tablet|               50.0|    200000.0|        Batal|Teknisi-TK009|           3.2|   Sesuai Target|                4000.0|
|     TK008| SVS-1002|  Cabang Semarang|         tablet|               62.0|    350000.0|      Selesai|Teknisi-TK008|           4.0| Me

In [27]:
# Analisis denga Window Function
from pyspark.sql.window import Window
from pyspark.sql.functions import rank, row_number, count, sum as spark_sum

df_sesuai = df_gabungan.filter(col("kategori_layanan") == "Sesuai Target").groupBy("cabang", "nama_teknisi").agg(count("servis_id").alias("jumlah"))

window_a = Window.partitionBy(col("cabang")).orderBy(col("jumlah").desc())

df_top3 = df_sesuai.withColumn("peringkat", rank().over(window_a)) \
    .filter(col("peringkat") < 4) \
    .select("cabang", "nama_teknisi", "jumlah", "peringkat")

df_top3.show()

+-----------------+-------------+------+---------+
|           cabang| nama_teknisi|jumlah|peringkat|
+-----------------+-------------+------+---------+
|  Cabang Magelang|Teknisi-TK010|    15|        1|
|  Cabang Magelang|Teknisi-TK001|    10|        2|
|  Cabang Magelang|Teknisi-TK016|     6|        3|
|  Cabang Semarang|Teknisi-TK004|    14|        1|
|  Cabang Semarang|Teknisi-TK015|    14|        1|
|  Cabang Semarang|Teknisi-TK017|    11|        3|
|Cabang Yogyakarta|Teknisi-TK003|    15|        1|
|Cabang Yogyakarta|Teknisi-TK014|    13|        2|
|Cabang Yogyakarta|Teknisi-TK007|    10|        3|
+-----------------+-------------+------+---------+



In [41]:
# Untuk SETIAP cabang, tentukan SATU teknisi dengan rating TERTINGGI
window_b = Window.partitionBy(col("cabang")).orderBy(col("rating_teknisi").desc())

best_teknisi = df_gabungan.withColumn(
    "rank", row_number().over(window_b)
).select(["cabang", "nama_teknisi", "rating_teknisi"]).where(col("rank") < 2).show()

+-----------------+-------------+--------------+
|           cabang| nama_teknisi|rating_teknisi|
+-----------------+-------------+--------------+
|  Cabang Magelang|Teknisi-TK001|           4.6|
|  Cabang Semarang|Teknisi-TK018|           4.1|
|Cabang Yogyakarta|Teknisi-TK011|           4.7|
+-----------------+-------------+--------------+



In [35]:
# Analisis dengan spark SQL
df_gabungan.createOrReplaceTempView("gabungan")

spark_sql = spark.sql('''
    SELECT cabang,
    COUNT (*) AS total_servis,
    SUM(CASE WHEN kategori_layanan = "Melebihi Target" THEN 1 ELSE 0 END) AS jumlah_melebihi_target,
    ROUND(
        (SUM(CASE WHEN kategori_layanan = "Melebihi Target" THEN 1 ELSE 0 END) * 100) / COUNT (*)
    ) AS presentase_melebihi_target
    FROM gabungan
    GROUP BY cabang
    ORDER BY presentase_melebihi_target DESC
''')

spark_sql.show()

+-----------------+------------+----------------------+--------------------------+
|           cabang|total_servis|jumlah_melebihi_target|presentase_melebihi_target|
+-----------------+------------+----------------------+--------------------------+
|Cabang Yogyakarta|         190|                   110|                      58.0|
|  Cabang Magelang|          86|                    50|                      58.0|
|  Cabang Semarang|         127|                    73|                      57.0|
+-----------------+------------+----------------------+--------------------------+



Cabang yang perlu mendapat perhatian manajemen adalah Cabang Yogyakarta. Hal ini dikarenakan cabang tersebut memiliki presetase melebihi target tertinggi yaitu di angkat 58% dengan 110 jumlah_melebihi_target dari 190 total servis. Artinya cabang ini dapat dinilai lambat dalam pengerjaan servis dikarenakan melebihi target penyelesaian servis yaitu di angka 60 menit

In [52]:
# Menyimpan hasil akhir
df_gabungan.write.mode("overwrite").partitionBy("cabang").parquet("hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis")

# Verifikasi hasil 
!hdfs dfs -ls "hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis"

Found 4 items
-rw-r--r--   3 mazedev supergroup          0 2026-10-08 10:08 hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis/_SUCCESS
drwxr-xr-x   - mazedev supergroup          0 2026-10-08 10:08 hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis/cabang=Cabang Magelang
drwxr-xr-x   - mazedev supergroup          0 2026-10-08 10:08 hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis/cabang=Cabang Semarang
drwxr-xr-x   - mazedev supergroup          0 2026-10-08 10:08 hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis/cabang=Cabang Yogyakarta


In [53]:
# Menampilkan jumlah baris setelah dirubah ke parquet
parquet = spark.read.parquet("hdfs://localhost:9000/user/mazedev/uts/processed/laporan_servis")

print("Jumlah baris:", parquet.count())

Jumlah baris: 403


In [29]:
spark.stop()